# 04 - Evaluation (Member 4)
Same test images, same metric code, same image range for all models. Report real and synthetic test separately.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, yaml
REPO = "/content/nighttime-flare-removal"
if not os.path.exists(REPO):
    !git clone https://github.com/USERNAME/nighttime-flare-removal.git {REPO}   # <-- replace USERNAME
%cd {REPO}
!git pull
!pip -q install pyyaml scikit-image tqdm
sys.path.insert(0, REPO)
cfg = yaml.safe_load(open("configs/default.yaml"))
import torch
print("CUDA:", torch.cuda.is_available())

In [ ]:
import os
from src.evaluation.evaluate import load_checkpoint, compare, save_comparison_grid
from src.data.dataset import PairedFolderDataset
from torch.utils.data import DataLoader
PROJECT = cfg["project"]["drive_root"]
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TEST DATA: point to the official REAL test pairs (never used for training/tuning)
TEST_INPUT, TEST_TARGET = "/content/flare_test/input", "/content/flare_test/target"   # <-- set
test_ds = PairedFolderDataset(TEST_INPUT, TEST_TARGET, crop_size=256, train=False)
test_loader = DataLoader(test_ds, batch_size=cfg["eval"]["batch_size"], shuffle=False)

In [ ]:
ck = PROJECT + "/checkpoints"
models = {
  "Plain U-Net": load_checkpoint("unet", ck + "/baseline/unet_best.pth", cfg, dev),
  "Attention + multiscale": load_checkpoint("attention_multiscale", ck + "/improved/attention_multiscale_best.pth", cfg, dev),
}
table = compare(models, test_loader, dev, out_json=PROJECT + "/results/metrics/test_metrics.json", use_lpips=False)

In [ ]:
save_comparison_grid(models, test_loader, dev, PROJECT + "/results/figures/comparison_grid.png", n=4)
from IPython.display import Image, display
display(Image(PROJECT + "/results/figures/comparison_grid.png"))

## Failure cases
Inspect and note: remaining flare, oversmoothing, colour shift, damaged light sources. Save selected figures to `results/figures/`.